In [2]:
# Hyperparameter Tuning
def hyperparameter_tunning(modol: object, model_name: str = 'LogisticRegression') -> object:

    # LogisticRegression classification model grid search
    if model_name == 'LogisticRegression':
        paramGrid = (
            ParamGridBuilder()
            .addGrid(modol.regParam, [0.01, 0.1, 0.2,0.3])        # regularization
            .addGrid(modol.elasticNetParam, [0.1, 0.5, 1.0])  # L2 vs L1
            .addGrid(modol.maxIter, [50,100, 200])
            .build()
        )
    
    # Gradient Boost Tree classification model grid search 
    if model_name == 'GBT':        
        paramGrid = (
            ParamGridBuilder()
            .addGrid(modol.maxIter, [100, 150])
            .addGrid(modol.maxDepth, [5, 7])
            .addGrid(modol.stepSize, [0.1, 0.05])
            .addGrid(modol.subsamplingRate, [0.7, 0.8])
            .build()
        )



    if model_name == 'RandomForest':
        
        paramGrid = (
            ParamGridBuilder()
             .addGrid(modol.maxDepth, [10, 17, 20])    # 5 → 20
             .addGrid(modol.maxBins, [10, 20, 30])
             .addGrid(modol.numTrees, [50, 100, 150])        # numTrees → maxIter
             .build()
        )

    
    # LightGBM classification model grid search : may not necessary because it performs better with default values
    if model_name == 'LightGBM':
        paramGrid = (
            ParamGridBuilder()
            .addGrid(modol.numLeaves, [31, 64])
            .addGrid(modol.learningRate, [0.05, 0.1])
            .addGrid(modol.numIterations, [100, 200])
            .build()
        )

    #xgboostclassifier
    if model_name == 'xgboost':
        paramGrid = (
            ParamGridBuilder()
            .addGrid(xgb.max_depth, [5, 7])
            .addGrid(xgb.n_estimators, [50, 100])   # <-- equivalent to n_estimators
            #.addGrid(xgb.subsample, [0.8, 1.0])
            #.addGrid(xgb.colsample_bytree, [0.8, 1.0])
            .build()
        )

    

    return paramGrid



StatementMeta(, 698b7736-87e0-47e9-8e09-c7e61c119afd, 4, Finished, Available, Finished, False)

In [5]:

def evaluate_and_validate(label_col: str, pipeline: object, paramGrid: object ):
    evaluator = BinaryClassificationEvaluator(
        labelCol=label_col,
        metricName="areaUnderROC"
    )

    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        numFolds=3,
        parallelism=4
    )
    return evaluator, cv

StatementMeta(, fa9ac7bb-a903-4981-ab62-ec6886379b90, 8, Finished, Available, Finished, False)

In [2]:
def encode_categorical_features(cols: list[str]) -> list:
    indexers = [
        StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep"
        )
        for c in cols
    ]

    encoders = [
        OneHotEncoder(
            inputCol=f"{c}_idx",
            outputCol=f"{c}_ohe"
        )
        for c in cols
    ]

    return indexers, encoders

StatementMeta(, fc36528e-1cd9-479f-be8b-c263538f6c04, 4, Finished, Available, Finished, False)

In [1]:
# Auto-detect column types
def auto_detect_colum_type(df, label_col) -> list:     

    categorical_cols = [
        c for (c, t) in spark_df.dtypes if t == "string" and c != label_col
    ]

    numeric_cols = [
        c for (c, t) in spark_df.dtypes
        if t in ("int", "double", "float", "bigint") and c not in [label_col, 'weight']
    ]

    print("Categorical:", categorical_cols)
    print("Numeric:", numeric_cols)

    return categorical_cols, numeric_cols

StatementMeta(, 69ff45a6-8777-40fc-9991-f65d15e8f867, 3, Finished, Available, Finished, False)

In [ ]:
def read_and_train_test_split():
    # -------------------------------------------
    # a. Read Dataset and create Spark DataFrame
    # -------------------------------------------

    spark_df = spark.read.table('FraudDetection.dbo.silver_layer_ciferfrauddata_2')

    # ** 4. Train / Test Split **
    train_df, test_df = spark_df.randomSplit([0.8, 0.2], seed=42)

    train_df.cache()
    test_df.cache()

In [ ]:
def decitionTree_model_experment(trial, train_df, test_df):
    max_depth = trial.suggest_int("max_depth", 5, 20)
    max_bins = trial.suggest_categorical("max_bins",[10, 20, 30])
    num_trees = trial.suggest_categorical("numTrees", [50, 100, 150])


    with mlflow.start_run(nested=True):
        #Train a model using the provided hyperparameter value
        #numericalFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
        #assembler = VectorAssembler(inputCols=numericalFeatures, outputCol="numericFeatures") # Numerical verctorized data
        #scaler = MinMaxScaler(inputCol=assembler.getOutputCol(), outputCol="normalizedFeatures")
        #featureVector = VectorAssembler(inputCols=["normalizedFeatures"], outputCol="features")

        # -----------------------------------------
        # 0. Get categorical and numerical features
        # --------------------------------------------
        label_col = "isFraud" 
        categorical_cols, numeric_cols = auto_detect_colum_type(spark_df, label_col)

        # **1. Preprocessing the pipeline** 
        # ---------------------------------------
        # 1.1 Encode categorical freatures 
        # --------------------------------------
        #indexers, encoders = encode_categorical_features(categorical_cols)
        indexers = [
            StringIndexer(
                inputCol=c,
                outputCol=f"{c}_idx",
                handleInvalid="keep"
            )
            for c in categorical_cols
        ]

        encoders = [
            OneHotEncoder(
                inputCol=f"{c}_idx",
                outputCol=f"{c}_ohe"
            )
            for c in categorical_cols
        ]

        # -------------------------------------------------------
        # 1.2 Assemble features 
        # ----------------------------
        assembler_inputs = [f"{c}_idx" for c in categorical_cols] + numeric_cols
        print('Assembler Inputs: ', assembler_inputs)

        assembler = VectorAssembler(
            inputCols=assembler_inputs,
            outputCol="features_raw"
        )

        # ----------------------------
        # 1.3 Scaling: MimMax scaling 
        # ----------------------------
        scaler = MinMaxScaler(
            inputCol="features_raw",
            outputCol="features"
        )
        
            
        dtAlgo = RandomForestClassifier(
            labelCol=label_col,
            featuresCol="features_raw" 

        ) #DecisionTreeClassifier(labelCol=label_col, featuresCol="features", maxDepth=max_depth, maxBins=max_bins)

        pipeline = Pipeline(stages=[*indexers, *encoders,  assembler, scaler, dtAlgo]) # Pipeline

        # ** 4. Hyperparamer tunning: for  **
        paramGrid = hyperparameter_tunning(dtAlgo, 'RandomForest')

        # ** 5. Evaluate and validate the model **
        evaluator, cv = evaluate_and_validate(label_col, pipeline, paramGrid)
            
        model = pipeline.fit(train_df)

        # Evaluate the model
        predictions = model.transform(test_df)

        eval = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")
        accuracy_score = eval.evaluate(predictions)

        # Log parameters and metrics
        mlflow.log_param('MaxDepth', max_depth)
        mlflow.log_param('MaxBins', max_bins)
        mlflow.log_metric('accuracy', accuracy_score)


        ## Infer and log model signiture
        #signature = infer_signature(train_df.select(numericalFeatures).toPandas(), predictions.select('prediction').toPandas())
        #mlflow.spark.log_model(model, "model", signature=signature, dfs_tmpdir='/Volumes/mlpractice/source/mlmodel/ml_lab/tmp/')
        
    return accuracy_score